# Evaluating RAG / Agent Quality with Strands Evals

How do you know if your RAG app or agent is actually *good*? You measure it. The slides introduce **Ragas** and its metrics vocabulary - context precision, context recall, faithfulness, response relevancy, and so on. Those concepts are what matter; the tooling is a means to an end.

For the runnable demo we use **Strands Evals** (`strands-agents-evals`), AWS's evaluation framework for Strands agents. It's the natural fit here for two reasons:

> **Teaching/Learning Tip:** (1) this course standardizes on **Strands**, so evaluating with the Strands-native tool keeps one coherent stack; (2) at the time of writing, Ragas's latest release is broken on import against modern LangChain, whereas Strands Evals installs and runs cleanly. Same *ideas* as the slides - LLM-as-a-judge scoring of answer quality - just a tool that works with our stack.

Requires `strands-agents` and `strands-agents-evals` (in `requirements.txt`).

## The evaluation dataset (maps to the Ragas `EvaluationDataset` slide)

Ragas builds a dataset of `SingleTurnSample`s (user_input, response, reference, ...). In Strands Evals the equivalent is a list of `Case`s: an `input` and an `expected_output` (the reference answer).

> **Teaching/Learning Tip:** an evaluation dataset is just questions paired with known-good answers. It's the ruler you measure the system against - the same role a test suite plays for ordinary code.

In [ ]:
# Strands Evals runs evaluations on an asyncio loop. Jupyter already runs one,
# so nest_asyncio lets the nested loop work inside the notebook.
import nest_asyncio
nest_asyncio.apply()

from strands import Agent
from strands.models import BedrockModel
from strands_evals import eval_task, Case, Experiment
from strands_evals.evaluators import OutputEvaluator

REGION = "us-east-1"
model = BedrockModel(model_id="us.amazon.nova-lite-v1:0", region_name=REGION)

# The evaluation dataset: questions paired with reference answers
cases = [
    Case[str, str](
        name="germany",
        input="What is the capital of Germany?",
        expected_output="Berlin is the capital of Germany.",
    ),
    Case[str, str](
        name="austen",
        input="Who wrote 'Pride and Prejudice'?",
        expected_output="Jane Austen wrote 'Pride and Prejudice'.",
    ),
    Case[str, str](
        name="superbowl",
        input="When was the first Super Bowl?",
        expected_output="The first Super Bowl was held on January 15, 1967.",
    ),
]
print(f"{len(cases)} evaluation cases")

## The system under test

We evaluate a Strands `Agent`. The `@eval_task()` decorator handles the boilerplate: for each case it runs the agent on `case.input` and captures the output. This stands in for whatever RAG app or agent you're actually testing.

In [ ]:
@eval_task()
def system_under_test():
    return Agent(
        model=model,
        system_prompt="You are a helpful assistant that answers factual questions concisely.",
        callback_handler=None,
    )

## Scoring with an LLM judge (maps to the Ragas metric slides)

Ragas metrics like `Faithfulness`, `ResponseRelevancy`, and correctness use an **evaluator LLM** to judge each answer. Strands Evals works the same way: the `OutputEvaluator` takes a judge `model` and a `rubric` describing what a good answer looks like, then scores each case.

> **Teaching/Learning Tip:** this is *LLM-as-a-judge* - one model grades another's output against criteria. The rubric is where you encode what you care about (accuracy, relevance, faithfulness to context). It's flexible but not deterministic, so judge scores are a signal, not gospel.

In [ ]:
evaluator = OutputEvaluator(
    model=model,
    rubric=(
        "Judge the answer against the expected answer.\n"
        "Score 1.0 if it is factually correct and relevant to the question.\n"
        "Score 0.5 if partially correct or missing key detail.\n"
        "Score 0.0 if incorrect or off-topic."
    ),
    include_inputs=True,
)

experiment = Experiment[str, str](cases=cases, evaluators=[evaluator])
report = experiment.run_evaluations(system_under_test)
print("Evaluation complete.")

## Read the results

The report carries parallel lists indexed by case: the score, pass/fail, and the judge's reasoning for each.

In [ ]:
for i, case in enumerate(report.cases):
    print(f"[{case.get('name')}]  score={report.scores[i]}  passed={report.test_passes[i]}")
    print(f"  reason: {report.reasons[i]}")
    print()

pass_rate = sum(report.test_passes) / len(report.test_passes)
print(f"Overall pass rate: {pass_rate:.0%}   average score: {report.overall_score:.2f}")

## Mapping the Ragas slides to Strands Evals

The slide metrics have direct or near analogs here - the concepts transfer even though the tool differs:

| Ragas metric (slides) | Strands Evals analog |
| --- | --- |
| Faithfulness | `FaithfulnessEvaluator` |
| Response relevancy | `ResponseRelevanceEvaluator` |
| Reference-based correctness | `CorrectnessEvaluator` |
| (answer quality, general) | `OutputEvaluator` (used above) |
| Tool / retrieval trajectory | `TrajectoryEvaluator` |

> **Teaching/Learning Tip:** the richer evaluators (`Faithfulness`, `Correctness`, `Trajectory`) are *trace-based* - they inspect the agent's execution trace, not just its final string, so they need a bit more setup (a traced agent run). `OutputEvaluator` is the simplest starting point and covers the core "is this answer good?" question the slides open with. Start simple, add trajectory evaluation when you need to see *how* the agent got there.